# **SUBSCRIBER**

In [5]:
import json
import os
import csv
import time
import uuid
from paho.mqtt import client as mqtt_client


# ============================================================
# TODO: UPDATE FOLDER PATH and OUTPUT FILE-NAME IN HERE
# ============================================================
SAVE_FOLDER = "data"  # Update this to your desired folder path
CSV_FILENAME = "test.csv"  # Update this to your desired CSV filename
OVERWRITE_EXISTING = True  # True = ล้างไฟล์เก่าทิ้งก่อนรับข้อมูลชุดใหม่ของวันแข่ง | False = ต่อท้ายไฟล์เดิมและข้าม timestamp ซ้ำ
OUTPUT_CSV = os.path.join(SAVE_FOLDER, CSV_FILENAME)

STATE = {
    "writer_initialized": False,
    "fieldnames": None,
    "first_message_received": False,
    "last_msg_time": None,
    "seen_timestamps": set(),
    "saved_count": 0,
    "skipped_count": 0,
}

# ============================================================
# TODO: MQTT CONFIG 
# ============================================================
MQTT_CONFIG = {
    "BROKER": "192.168.1.1", 
    "PORT": 1883,
    "TOPIC": "idt/cpt/grh/publish-test",
    "CLIENT_ID": f"3B-{uuid.uuid4().hex[:8]}",
    "KEEPALIVE": 120,
}

def connect_mqtt():
    client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION2, MQTT_CONFIG["CLIENT_ID"])
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(MQTT_CONFIG["BROKER"], MQTT_CONFIG["PORT"], MQTT_CONFIG["KEEPALIVE"])
    return client


def on_connect(client, userdata, flags, reason_code, properties):
    if not reason_code.is_failure:
        print("Connected to MQTT broker.")
        client.subscribe(MQTT_CONFIG["TOPIC"], qos=1)
        print(f"Subscribed to topic: {MQTT_CONFIG['TOPIC']}")
    else:
        print(f"Failed to connect, reason code: {reason_code}")


def load_seen_timestamps():
    if not os.path.isfile(OUTPUT_CSV):
        return
    with open(OUTPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if STATE["fieldnames"] is None and reader.fieldnames:
            STATE["fieldnames"] = list(reader.fieldnames)
        for row in reader:
            ts = str(row.get("timestamp", "")).strip()
            if ts:
                STATE["seen_timestamps"].add(ts)


def save_data_csv(data: dict):
    if SAVE_FOLDER:
        os.makedirs(SAVE_FOLDER, exist_ok=True)

    if not STATE["writer_initialized"]:
        load_seen_timestamps()
        if STATE["fieldnames"] is None:
            STATE["fieldnames"] = list(data.keys())
        file_exists = os.path.isfile(OUTPUT_CSV)
        if not file_exists:
            with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])
                writer.writeheader()
        STATE["writer_initialized"] = True

    ts = str(data.get("timestamp", "")).strip()
    if ts and ts in STATE["seen_timestamps"]:
        STATE["skipped_count"] += 1
        print(f"Duplicate timestamp skipped: {ts}")
        return False

    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"], extrasaction="ignore")
        writer.writerow(data)

    if ts:
        STATE["seen_timestamps"].add(ts)
    STATE["saved_count"] += 1
    return True


def on_message(client, userdata, msg):
    STATE["first_message_received"] = True
    STATE["last_msg_time"] = time.time()

    try:
        payload_str = msg.payload.decode("utf-8")
        data = json.loads(payload_str)
        if not isinstance(data, dict):
            print("Skipped non-object JSON payload.")
            return
        if save_data_csv(data):
            print(f"Saved row to {OUTPUT_CSV}: {data}")

    except Exception as e:
        print(f"Error processing message: {e}")


def reset_state():
    STATE["writer_initialized"] = False
    STATE["fieldnames"] = None
    STATE["first_message_received"] = False
    STATE["last_msg_time"] = None
    STATE["seen_timestamps"] = set()
    STATE["saved_count"] = 0
    STATE["skipped_count"] = 0


def subscriber(IDLE_TIMEOUT = 10):
    reset_state()
    if OVERWRITE_EXISTING and os.path.isfile(OUTPUT_CSV):
        os.remove(OUTPUT_CSV)
        print(f"Removed previous file before fresh collection: {OUTPUT_CSV}")

    client = connect_mqtt()
    print(f"Starting MQTT Subscribe. Topic={MQTT_CONFIG['TOPIC']} -> {OUTPUT_CSV} (Timeout={IDLE_TIMEOUT}s)")

    client.loop_start()
    try:
        while True:
            if STATE["first_message_received"]:
                idle = time.time() - STATE["last_msg_time"]
                if idle >= IDLE_TIMEOUT:
                    print(
                        f"No Subscribe message for {IDLE_TIMEOUT}s -- stop subscribe process."
                    )
                    break
            time.sleep(0.5)

    except KeyboardInterrupt:
        print("\nStopping subscriber manually...")

    finally:
        client.loop_stop()
        client.disconnect()
        print(f"Summary -> Saved: {STATE['saved_count']} rows | Skipped duplicates: {STATE['skipped_count']} | File: {OUTPUT_CSV}")
        print("Disconnected from MQTT broker.")


if __name__ == "__main__":
    subscriber()


Starting MQTT Subscribe. Topic=idt/cpt/grh/publish-test -> data\test.csv (Timeout=10s)
Connected to MQTT broker.
Subscribed to topic: idt/cpt/grh/publish-test
Saved row to data\test.csv: {'timestamp': '2026-10-15 09:00:51', 'temperature_celsius': 20.0300660630505, 'humidity_percentage': 44.20565550797552, 'lux_lumen': 36925.31144172551, 'CO2_ppm': 1003.3294620053414, 'y': 0, 'y_label': 'Normal_state'}
Saved row to data\test.csv: {'timestamp': '2026-10-15 09:00:52', 'temperature_celsius': 29.206435717491196, 'humidity_percentage': 43.149173487267205, 'lux_lumen': 27369.184507743485, 'CO2_ppm': 604.4241549752337, 'y': 0, 'y_label': 'Normal_state'}
Saved row to data\test.csv: {'timestamp': '2026-10-15 09:00:53', 'temperature_celsius': 28.68007189586376, 'humidity_percentage': 54.14391677447566, 'lux_lumen': 18018.580188192853, 'CO2_ppm': 1056.8063997962995, 'y': 0, 'y_label': 'Normal_state'}
Saved row to data\test.csv: {'timestamp': '2026-10-15 09:00:54', 'temperature_celsius': 22.6228392